# 🐷 Pig Posture Recognition – DINOv2 ViT-L Training

**Model:** DINOv2 ViT-Large (pretrained, `timm`)  
**Input:** 518×518 cropped pig instances  
**Metric:** Macro-averaged F1 score  
**Multi-GPU:** DataParallel on GPU 0, 1, 2  
**Key advantage:** Meta's self-supervised foundation model. The self-distillation pre-training produces features that generalize extremely well across domains – ideal for cross-view and cross-farm generalization in pig posture recognition.

## ⚙️ Configuration

In [6]:
TAG = "T2"   # "T1" or "T2"
DATA_ROOT = "/datasets/multi-view-pig-posture-recognition"

if TAG == "T1":
    CSV_PATH = f"{DATA_ROOT}/train1.csv"
    IMG_DIR  = f"{DATA_ROOT}/train1_images"
else:
    CSV_PATH = f"{DATA_ROOT}/train2.csv"
    IMG_DIR  = f"{DATA_ROOT}/train2_images"

OUTPUT_DIR = f"runs/dinov2_{TAG.lower()}"

MODEL_NAME    = "vit_large_patch14_reg4_dinov2.lvd142m"  # ~307M params
IMG_SIZE      = 518                  # must be divisible by 14 (patch size); 518 = 37 * 14
BATCH_SIZE    = 8
GRAD_ACCUM    = 4                    # effective batch = 32
EPOCHS        = 10
LR            = 5e-5
WARMUP_EPOCHS = 3
VAL_FRAC      = 0.10
LABEL_SMOOTH  = 0.10
MIXUP_ALPHA   = 0.40
CUTMIX_ALPHA  = 1.0
PAD_RATIO     = 0.25
NUM_WORKERS   = 8
SEED          = 42
NUM_CLASSES   = 5
CLASS_NAMES   = ["Lateral_lying_left", "Lateral_lying_right",
                 "Sitting", "Standing", "Sternal_lying"]

# Multi-GPU: welche GPUs verwenden
GPU_IDS       = [0, 1, 2]            # nur GPU 0, 1, 2

print(f"Tag: {TAG}  |  Model: {MODEL_NAME}  |  ImgSize: {IMG_SIZE}")
print(f"CSV: {CSV_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}  |  GPUs: {GPU_IDS}")

Tag: T2  |  Model: vit_large_patch14_reg4_dinov2.lvd142m  |  ImgSize: 518
CSV: /datasets/multi-view-pig-posture-recognition/train2.csv
Output: runs/dinov2_t2
Effective batch: 32  |  GPUs: [0, 1, 2]


## 📦 Install Dependencies

In [2]:
!pip install timm tqdm scikit-learn pandas pillow matplotlib -q

## 📚 Imports

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"   # nur GPU 0, 1, 2 sichtbar

import ast, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Available GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} – {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

<jemalloc>: Unsupported system page size


Device: cuda
Available GPUs: 3
  GPU 0: Tesla V100-SXM2-32GB – 33.8 GB
  GPU 1: Tesla V100-SXM2-32GB – 33.8 GB
  GPU 2: Tesla V100-SXM2-32GB – 33.8 GB


## 🔒 Reproducibility

In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 🔍 Load Data

In [5]:
df = pd.read_csv(CSV_PATH)
print(f"Total instances: {len(df)}  |  Unique images: {df['image_id'].nunique()}")
print("\nClass distribution:")
for c in range(NUM_CLASSES):
    cnt = (df["class_id"] == c).sum()
    bar = "█" * int(30 * cnt / len(df))
    print(f"  {c} - {CLASS_NAMES[c]:<22} {bar:<30} {cnt:>5}  ({100*cnt/len(df):.1f}%)")

Total instances: 23450  |  Unique images: 3150

Class distribution:
  0 - Lateral_lying_left     ███                             3083  (13.1%)
  1 - Lateral_lying_right    ████                            3435  (14.6%)
  2 - Sitting                                                 695  (3.0%)
  3 - Standing               ████████████                    9928  (42.3%)
  4 - Sternal_lying          ████████                        6309  (26.9%)


## 🗂️ Dataset & Augmentations

In [7]:
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, pad_ratio=0.25):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.pad_ratio = pad_ratio

    def __len__(self): return len(self.df)

    def _crop(self, img, bbox):
        W, H = img.size
        x, y, w, h = [float(v) for v in ast.literal_eval(bbox)]
        px, py = w * self.pad_ratio, h * self.pad_ratio
        x1 = max(0, int(x - px)); y1 = max(0, int(y - py))
        x2 = min(W, int(x+w+px)); y2 = min(H, int(y+h+py))
        return img.crop((x1, y1, x2, y2))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        crop = self._crop(img, row["bbox"])
        if self.transform: crop = self.transform(crop)
        return crop, int(row["class_id"])


MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def get_train_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size + 64, size + 64)),
        T.RandomCrop(size),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.3),
        T.RandomRotation(degrees=30),
        T.RandomAffine(degrees=0, scale=(0.80, 1.20)),
        T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.10),
        T.RandomGrayscale(p=0.1),
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.3, scale=(0.02, 0.25)),
    ])

def get_val_transform(size=IMG_SIZE):
    return T.Compose([
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

print(f"Dataset and transforms defined. IMG_SIZE={IMG_SIZE} (must be multiple of 14 for ViT-14).")

Dataset and transforms defined. IMG_SIZE=518 (must be multiple of 14 for ViT-14).


## ✂️ Train / Validation Split

In [8]:
train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, stratify=df["class_id"], random_state=SEED
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

train_ds = PigPostureDataset(train_df, IMG_DIR, transform=get_train_transform(), pad_ratio=PAD_RATIO)
val_ds   = PigPostureDataset(val_df,   IMG_DIR, transform=get_val_transform(),   pad_ratio=PAD_RATIO)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print("DataLoaders ready.")

Train: 21105  |  Val: 2345
DataLoaders ready.


## 🧠 Model, Loss & Optimizer (Multi-GPU)

In [9]:
# DINOv2 ViT-L/14 with 4 register tokens
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

# ─── Multi-GPU: auf GPU 0, 1, 2 verteilen ───
model = nn.DataParallel(model, device_ids=GPU_IDS)
print(f"✓ DataParallel aktiv auf {len(GPU_IDS)} GPUs: {GPU_IDS}")

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_NAME}  |  Parameters: {total_params:.1f}M")

counts  = Counter(train_df["class_id"].tolist())
total   = len(train_df)
weights = torch.tensor(
    [total / (NUM_CLASSES * max(counts.get(c, 1), 1)) for c in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)
print("Class weights:", [f"{w:.2f}" for w in weights.cpu()])

criterion       = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTH)
criterion_plain = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05, betas=(0.9, 0.999))

def warmup_cosine_schedule(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine_schedule)
scaler    = GradScaler()
print("Ready.")

✓ DataParallel aktiv auf 3 GPUs: [0, 1, 2]
Model: vit_large_patch14_reg4_dinov2.lvd142m  |  Parameters: 304.4M
Class weights: ['1.52', '1.37', '6.74', '0.47', '0.74']
Ready.


## 🛠️ Helper Functions (MixUp + CutMix + Gradient Accumulation)

In [10]:
def mixup_data(x, y, alpha=0.4):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def cutmix_data(x, y, alpha=1.0):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    B, C, H, W = x.shape
    cut_ratio = np.sqrt(1 - lam)
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)
    x_new = x.clone()
    x_new[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return x_new, y, y[idx], lam

def mixup_cutmix_data(x, y, mixup_alpha=0.4, cutmix_alpha=1.0):
    if np.random.rand() < 0.5:
        return mixup_data(x, y, mixup_alpha)
    return cutmix_data(x, y, cutmix_alpha)

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_sum, preds_all, labels_all = 0.0, [], []
    optimizer.zero_grad()
    for batch_idx, (imgs, labels) in enumerate(tqdm(loader, desc="  Train", leave=False)):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs, y_a, y_b, lam = mixup_cutmix_data(imgs, labels, MIXUP_ALPHA, CUTMIX_ALPHA)
        with autocast():
            logits = model(imgs)
            loss   = mixup_loss(criterion_plain, logits, y_a, y_b, lam) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
        loss_sum += loss.item() * GRAD_ACCUM * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(y_a.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0)


@torch.no_grad()
def validate_epoch(model, loader):
    model.eval()
    loss_sum, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(loader, desc="  Val  ", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        loss_sum += loss.item() * imgs.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
    n = len(loader.dataset)
    return loss_sum / n, f1_score(labels_all, preds_all, average="macro", zero_division=0), preds_all, labels_all

## 🚀 Training Loop

In [ ]:
best_val_f1 = 0.0
log         = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1   = train_one_epoch(model, train_loader, optimizer, scaler)
    val_loss, val_f1, _, _ = validate_epoch(model, val_loader)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    mark = "★" if val_f1 > best_val_f1 else " "
    print(f"{mark} Epoch {epoch:03d}/{EPOCHS} | "
          f"Train loss={train_loss:.4f} f1={train_f1:.4f} | "
          f"Val loss={val_loss:.4f} f1={val_f1:.4f} | lr={lr:.2e}")

    log.append(dict(epoch=epoch, train_loss=train_loss, train_f1=train_f1,
                    val_loss=val_loss, val_f1=val_f1, lr=lr))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        ckpt_path   = os.path.join(OUTPUT_DIR, "best_model.pth")
        # .module.state_dict() weil DataParallel das Modell wrappet
        torch.save({"epoch": epoch, "model": model.module.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "val_f1": val_f1, "model_name": MODEL_NAME, "tag": TAG}, ckpt_path)
        print(f"  → Saved best model (val_f1={val_f1:.4f})")

print(f"\n✅ Training complete. Best Val F1: {best_val_f1:.4f}")
pd.DataFrame(log).to_csv(os.path.join(OUTPUT_DIR, "training_log.csv"), index=False)

  Train:   0%|          | 0/2638 [00:00<?, ?it/s]

  Val  :   0%|          | 0/294 [00:00<?, ?it/s]

★ Epoch 001/10 | Train loss=1.0698 f1=0.4872 | Val loss=1.5161 f1=0.8433 | lr=3.33e-05
  → Saved best model (val_f1=0.8433)


  Train:   0%|          | 0/2638 [00:00<?, ?it/s]

  Val  :   0%|          | 0/294 [00:00<?, ?it/s]

...

  Epoch 004/10 | Train loss=0.8045 f1=0.6018 | Val loss=1.5851 f1=0.8666 | lr=4.75e-05


  Train:   0%|          | 0/2638 [00:00<?, ?it/s]

  Val  :   0%|          | 0/294 [00:00<?, ?it/s]

★ Epoch 005/10 | Train loss=0.7736 f1=0.6197 | Val loss=1.4627 f1=0.8999 | lr=4.06e-05
  → Saved best model (val_f1=0.8999)


  Train:   0%|          | 0/2638 [00:00<?, ?it/s]

## 📈 Training Curves

In [ ]:
import matplotlib.pyplot as plt

log_df = pd.DataFrame(log)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(log_df["epoch"], log_df["train_loss"], label="Train")
axes[0].plot(log_df["epoch"], log_df["val_loss"],   label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title(f"DINOv2 ViT-L {TAG} – Loss"); axes[0].legend()

axes[1].plot(log_df["epoch"], log_df["train_f1"], label="Train")
axes[1].plot(log_df["epoch"], log_df["val_f1"],   label="Val")
axes[1].axhline(best_val_f1, color="red", ls="--", alpha=0.6, label=f"Best={best_val_f1:.4f}")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro F1")
axes[1].set_title(f"DINOv2 ViT-L {TAG} – F1"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=120)
plt.show()

## 📊 Classification Report

In [ ]:
best_ckpt = torch.load(os.path.join(OUTPUT_DIR, "best_model.pth"), map_location=DEVICE)
model.module.load_state_dict(best_ckpt["model"])
_, _, val_preds, val_labels = validate_epoch(model, val_loader)
print(classification_report(val_labels, val_preds, target_names=CLASS_NAMES))